In [ ]:
import ee
import geemap

# Initialize the Earth Engine library.
try:
    ee.Initialize(project='project-183fe1c7-ddb0-4eea-8c2')
except Exception:
    print("Earth Engine not authenticated. Starting authentication flow...")
    ee.Authenticate()
    ee.Initialize(project='project-183fe1c7-ddb0-4eea-8c2')

In [ ]:
def get_s2_sr_cld_col(aoi, start_date, end_date):
    """Build a Sentinel-2 SR collection joined with s2cloudless probability."""
    s2_sr_col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
    )

    s2_cloudless_col = (
        ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
    )

    return ee.ImageCollection(
        ee.Join.saveFirst('s2cloudless').apply(
            primary=s2_sr_col,
            secondary=s2_cloudless_col,
            condition=ee.Filter.equals(
                leftField='system:index',
                rightField='system:index',
            ),
        )
    )


def add_cloud_bands(img):
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')
    return img.addBands(ee.Image([cld_prb, is_cloud]))


def add_shadow_bands(img):
    not_water = img.select('SCL').neq(6)
    dark_pixels = (
        img.select('B8')
        .lt(NIR_DRK_THRESH * SR_BAND_SCALE)
        .multiply(not_water)
        .rename('dark_pixels')
    )

    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))
    cld_proj = (
        img.select('clouds')
        .directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST * 10)
        .reproject(crs=img.select(0).projection(), scale=100)
        .select('distance')
        .mask()
        .rename('cloud_transform')
    )

    shadows = cld_proj.multiply(dark_pixels).rename('shadows')
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))


def add_cld_shdw_mask(img):
    img_cloud = add_cloud_bands(img)
    img_cloud_shadow = add_shadow_bands(img_cloud)
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)
    is_cld_shdw = (
        is_cld_shdw.focalMin(2)
        .focalMax(BUFFER * 2 / 20)
        .reproject(crs=img.select([0]).projection(), scale=20)
        .rename('cloudmask')
    )
    return img_cloud_shadow.addBands(is_cld_shdw)


def apply_cld_shdw_mask(img):
    not_cld_shdw = img.select('cloudmask').Not()
    return img.select('B.*').updateMask(not_cld_shdw)

In [ ]:
# Define collection filter and s2cloudless cloud-shadow mask parameters.
AOI = ee.Geometry.Point([83.277, 17.7009]).buffer(20000)
START_DATE = '2020-01-01'
END_DATE = '2020-12-31'
CLOUD_FILTER = 80
CLD_PRB_THRESH = 35
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 2
BUFFER = 100
SR_BAND_SCALE = 1e4

dataset = get_s2_sr_cld_col(AOI, START_DATE, END_DATE)

cloudless_image = (
    dataset.map(add_cld_shdw_mask)
    .map(apply_cld_shdw_mask)
    .median()
    .divide(SR_BAND_SCALE)
    .clip(AOI)
)

In [ ]:
# Visualization parameters
visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}

# Create a map object using geemap
Map = geemap.Map()

# Set center of the map
Map.setCenter(83.277, 17.7009, 12)

# Add the cloudless median composite to the map
Map.addLayer(cloudless_image, visualization, 'Sentinel-2 Cloudless RGB')

# test

In [ ]:
# Display the map in the notebook
Map